In [7]:
import numpy as np 
import os
import sys
from osgeo import gdal, ogr, osr
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
sns.set_theme()

import fastkml
from shapely import Polygon, box
import geopandas as gpd

# ====== import my own functions =========
import sys
import os
scripts_dir = (os.path.dirname(os.getcwd()) + '/scripts/')
sys.path.append(scripts_dir)

from layers import assemble_data

In [2]:
data_dir = '/bsuhome/julialober/scratch/coherence_data/'

mcs_aoi = gpd.read_file('/bsuhome/julialober/scratch/coherence_data/kmls/mores_creek.shp')
mcs_aoi

,id,geometry
0,1,"POLYGON ((-115.6739 43.98328, -115.67315 43.98..."


In [3]:
aoi = box(*mcs_aoi.total_bounds)
dates = [("20200101", "20200113"), 
         ("20200101", "20200129"),
         ("20200113", "20200129"),
         ("20210202", "20210214")]
fp_out = data_dir + 'stacks/'

In [24]:
import s3fs
import fsspec
base_url = f's3://noaa-nws-aorc-v1-1-1km'
years = {2020}
s3_out = s3fs.S3FileSystem(anon=True)
# print(years)
fileset = [s3fs.S3Map(
                root=f"s3://{base_url}/{yr}.zarr", s3=s3_out, check=False
            ) for yr in years]
print(fileset)
ds_2020 = xr.open_mfdataset(fileset, engine='zarr')
print(ds_2020)

<xarray.Dataset> Size: 20TB
Dimensions:              (time: 8784, latitude: 4201, longitude: 8401)
Coordinates:
  * latitude             (latitude) float64 34kB 20.0 20.01 20.02 ... 54.99 55.0
  * longitude            (longitude) float64 67kB -130.0 -130.0 ... -60.01 -60.0
  * time                 (time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31...
Data variables:
    APCP_surface         (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DLWRF_surface        (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DSWRF_surface        (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    PRES_surface         (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    SPFH_2maboveground   (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    TMP_2maboveground    (tim

In [14]:
years = {2021}
s3_out = s3fs.S3FileSystem(anon=True)
# print(years)
fileset = [s3fs.S3Map(
                root=f"s3://{base_url}/{yr}.zarr", s3=s3_out, check=False
            ) for yr in years]
print(fileset)
ds_2021 = xr.open_mfdataset(fileset, engine='zarr')
print(ds_2021)

<xarray.Dataset> Size: 20TB
Dimensions:              (time: 8760, latitude: 4201, longitude: 8401)
Coordinates:
  * latitude             (latitude) float64 34kB 20.0 20.01 20.02 ... 54.99 55.0
  * longitude            (longitude) float64 67kB -130.0 -130.0 ... -60.01 -60.0
  * time                 (time) datetime64[ns] 70kB 2021-01-01 ... 2021-12-31...
Data variables:
    APCP_surface         (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DLWRF_surface        (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DSWRF_surface        (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    PRES_surface         (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    SPFH_2maboveground   (time, latitude, longitude) float64 2TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    TMP_2maboveground    (tim

In [22]:
merged = xr.concat([ds_2020, ds_2021], dim='time')
merged

<xarray.Dataset> Size: 40TB
Dimensions:              (time: 17544, latitude: 4201, longitude: 8401)
Coordinates:
  * latitude             (latitude) float64 34kB 20.0 20.01 20.02 ... 54.99 55.0
  * longitude            (longitude) float64 67kB -130.0 -130.0 ... -60.01 -60.0
  * time                 (time) datetime64[ns] 140kB 2020-01-01 ... 2021-12-3...
Data variables:
    APCP_surface         (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DLWRF_surface        (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    DSWRF_surface        (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    PRES_surface         (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    SPFH_2maboveground   (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    TMP_2maboveground    (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    UGRD_10maboveground  (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>
    VGRD_10maboveground  (time, latitude, longitude) float64 5TB dask.array<chunksize=(144, 128, 256), meta=np.ndarray>

In [4]:
data = assemble_data(aoi,
                     dates,
                     fp_out)
data

/bsuhome/julialober/miniforge3/envs/coherence/lib/python3.13/site-packages/pygeoutils/pygeoutils.py:300: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge(


Got NLCD: <class 'xarray.core.dataset.Dataset'>
Making call to py3dep.get_map with resolution 30 in m.
Got DEM: <class 'xarray.core.dataarray.DataArray'>
{2020, 2021}
[<fsspec.mapping.FSMap object at 0x2aaae7ac5fd0>, <fsspec.mapping.FSMap object at 0x2aab05539e50>]


ValueError: Coordinate variable time is neither monotonically increasing nor monotonically decreasing on all datasets

In [ ]:
data.attrs